In [1]:
!pip install -q transformers==4.46.3 tokenizers==0.20.3 addict easydict einops kagglehub
import transformers; print(transformers.__version__)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 77.6 MB/s eta 0:00:00:00:01:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.0/3.0 MB 85.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 38.4 MB/s eta 0:00:00
4.46.3


In [2]:
import kagglehub
DATA_DL = kagglehub.dataset_download("sarvarbekerniyazov/ocripple-images")
print("downloaded to:", DATA_DL)

import os
for dirpath, dirs, files in os.walk(DATA_DL):
    depth = dirpath.replace(DATA_DL, "").count("/")
    if depth <= 2:
        print("  "*depth + dirpath.split("/")[-1] + f"/ ({len(files)} files)")

downloaded to: /kaggle/input/datasets/sarvarbekerniyazov/ocripple-images
ocripple-images/ (0 files)
  arxiv_scans/ (1 files)
    gt/ (43 files)
    clean/ (43 files)
    light/ (43 files)
    heavy/ (43 files)
    medium/ (43 files)
  mpdocvqa/ (2 files)
    pages/ (350 files)


In [3]:
!pip install -q transformers==4.46.3 tokenizers==0.20.3 addict easydict einops kagglehub
import transformers
print("transformers:", transformers.__version__)

transformers: 4.46.3


In [4]:
import torch, kagglehub
from pathlib import Path

print("GPU:", torch.cuda.get_device_name(0))

DATA_DL = kagglehub.dataset_download("sarvarbekerniyazov/ocripple-images")
DATA = Path(DATA_DL)
print("DATA root:", DATA)

n_arxiv = sum(1 for _ in (DATA/"arxiv_scans").rglob("*.png"))
n_mp = sum(1 for _ in (DATA/"mpdocvqa"/"pages").glob("*.png"))
print(f"arxiv pngs: {n_arxiv}, mpdocvqa pages: {n_mp}")   # 172 va 350 kutiladi

GPU: Tesla T4
DATA root: /kaggle/input/datasets/sarvarbekerniyazov/ocripple-images
arxiv pngs: 172, mpdocvqa pages: 350


In [5]:
import time, json
from tqdm.auto import tqdm

OUT = Path("/kaggle/working/ocr_outputs")

def collect_images():
    tasks = []
    for lv in ["clean", "light", "medium", "heavy"]:
        for p in sorted((DATA/"arxiv_scans"/lv).glob("*.png")):
            tasks.append((p, f"arxiv/{lv}/{p.stem}"))
    for p in sorted((DATA/"mpdocvqa"/"pages").glob("*.png")):
        tasks.append((p, f"mpdocvqa/{p.stem}"))
    assert tasks, "0 images found!"
    return tasks

def run_engine(engine_name, page_fn):
    tasks = collect_images()
    print(f"{engine_name}: {len(tasks)} images")
    eng_dir = OUT/engine_name
    eng_dir.mkdir(parents=True, exist_ok=True)
    timing = []
    for img_path, stem in tqdm(tasks, desc=engine_name):
        out = eng_dir/f"{stem}.txt"
        out.parent.mkdir(parents=True, exist_ok=True)
        if out.exists():
            continue
        t0 = time.perf_counter()
        try:
            text = page_fn(img_path)
        except Exception as e:
            text = ""
            print("ERR", img_path.name, repr(e)[:120])
        timing.append({"image": img_path.name,
                       "sec": round(time.perf_counter()-t0, 3)})
        out.write_text(text or "", encoding="utf-8")
    (eng_dir/"timing.json").write_text(json.dumps(timing))
    print("done:", len(timing))

print("images total:", len(collect_images()))   # 522 kutiladi

images total: 522


In [6]:
from transformers import AutoModel, AutoTokenizer

MODEL_ID = "deepseek-ai/DeepSeek-OCR"
tok = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModel.from_pretrained(
    MODEL_ID, trust_remote_code=True,
    torch_dtype=torch.float16, device_map="cuda",
    _attn_implementation="eager").eval()
print("model loaded")

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/801 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

modeling_deepseekocr.py: 0.00B [00:00, ?B/s]

...

A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingface.co/deepseek-ai/DeepSeek-OCR:
- modeling_deepseekocr.py
- modeling_deepseekv2.py
- deepencoder.py
- conversation.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
You are using a model of type deepseek_vl_v2 to instantiate a model of type DeepseekOCR. This is not supported for all configurations of models and can yield errors.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-000001.safetensors:   0%|          | 0.00/6.67G [00:00<?, ?B/s]

Some weights of DeepseekOCRForCausalLM were not initialized from the model checkpoint at deepseek-ai/DeepSeek-OCR and are newly initialized: ['model.vision_model.embeddings.position_ids']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


model loaded


In [7]:
PROMPT = "<image>\nFree OCR."
TMP = "/kaggle/working/ds_tmp"

def deepseek_page(img_path):
    res = model.infer(tok, prompt=PROMPT, image_file=str(img_path),
                      output_path=TMP,
                      base_size=1024, image_size=640, crop_mode=True,
                      save_results=False, eval_mode=True)
    return res if isinstance(res, str) else str(res)

test_img = next((DATA/"arxiv_scans"/"clean").glob("*.png"))
out = deepseek_page(test_img)
print(repr(out[:400]))

/usr/local/lib/python3.12/dist-packages/transformers/generation/configuration_utils.py:590: UserWarning: `do_sample` is set to `False`. However, `temperature` is set to `0.0` -- this flag is only used in sample-based generation modes. You should set `do_sample=True` or unset `temperature`.
  warnings.warn(
The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.
The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
The `seen_tokens` attribute is deprecated and will be removed in v4.41. Use the `cache_position` model input instead.
`get_max_cache()` is deprecated for all Cache classes. Use `get_max_cache_shape()` instead. Cal

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])


RuntimeError: masked_scatter_: expected self and source to have same dtypes but got Half and Float

In [8]:
run_engine("deepseek_ocr", deepseek_page)

deepseek_ocr: 522 images


deepseek_ocr:   0%|          | 0/522 [00:00<?, ?it/s]

The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])
ERR attention_p000.png RuntimeError('masked_scatter_: expected self and source to have same dtypes but got Half and Float')


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


...

BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])
ERR zrhw0228_p8.png RuntimeError('masked_scatter_: expected self and source to have same dtypes but got Half and Float')


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])
ERR zrhw0228_p9.png RuntimeError('masked_scatter_: expected self and source to have same dtypes but got Half and Float')


The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
Setting `pad_token_id` to `eos_token_id`:None for open-end generation.


BASE:  torch.Size([1, 256, 1280])
PATCHES:  torch.Size([6, 100, 1280])
ERR zxfk0226_p12.png RuntimeError('masked_scatter_: expected self and source to have same dtypes but got Half and Float')
done: 522


In [9]:
import shutil
shutil.make_archive("/kaggle/working/unlimited_outputs", "zip", OUT)

'/kaggle/working/unlimited_outputs.zip'